# Part 3: Model Predictions & Analysis
## Test Your Trained Models on New Reviews

In this notebook, you can:
1. Load your trained models
2. Make predictions on new reviews
3. Analyze aspect-level sentiment
4. Test multilingual capabilities

---

## Setup

In [ ]:
import torch
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    XLMRobertaTokenizer, XLMRobertaForSequenceClassification
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Load Trained Models

Choose which model to use for predictions.

In [ ]:
# ⚠️ CHOOSE YOUR MODEL
MODEL_CHOICE = 'xlm-roberta-base'  # Options: 'bert-base-uncased', 'roberta-base', 'xlm-roberta-base'

print(f"Loading {MODEL_CHOICE}...")

# Load model and tokenizer
if 'xlm-roberta' in MODEL_CHOICE:
    tokenizer = XLMRobertaTokenizer.from_pretrained(f'./models/{MODEL_CHOICE}')
    model = XLMRobertaForSequenceClassification.from_pretrained(f'./models/{MODEL_CHOICE}')
elif 'roberta' in MODEL_CHOICE:
    tokenizer = RobertaTokenizer.from_pretrained(f'./models/{MODEL_CHOICE}')
    model = RobertaForSequenceClassification.from_pretrained(f'./models/{MODEL_CHOICE}')
else:
    tokenizer = BertTokenizer.from_pretrained(f'./models/{MODEL_CHOICE}')
    model = BertForSequenceClassification.from_pretrained(f'./models/{MODEL_CHOICE}')

model.to(device)
model.eval()

print(f"✓ {MODEL_CHOICE} loaded and ready!")

## Prediction Function

In [ ]:
def predict_sentiment(text, show_confidence=True):
    """
    Predict sentiment for a single review.
    
    Args:
        text (str): Review text
        show_confidence (bool): Show confidence scores
    
    Returns:
        str: Predicted sentiment
    """
    # Tokenize
    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=1)
        prediction = torch.argmax(logits, dim=1).item()
    
    label_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
    sentiment = label_map[prediction]
    
    if show_confidence:
        confidence = probabilities[0][prediction].item()
        print(f"\n  → Sentiment: {sentiment}")
        print(f"  → Confidence: {confidence:.2%}")
        print(f"  → Probabilities: Neg={probabilities[0][0]:.2%}, Neu={probabilities[0][1]:.2%}, Pos={probabilities[0][2]:.2%}")
    
    return sentiment

print("✓ Prediction function defined")

## Test with Example Reviews

In [ ]:
# Example reviews (English, French, German)
example_reviews = [
    "The food was absolutely amazing! Best pasta I've ever had. Service was great too.",
    "Terrible experience. The waiter was rude and the food was cold. Would not recommend.",
    "Decent restaurant. Nothing special but nothing bad either. Average prices.",
    "Great ambiance and reasonable prices, but the service was quite slow.",
    "La nourriture était délicieuse mais le service était très lent.",  # French
    "Das Essen war ausgezeichnet! Sehr empfehlenswert.",  # German
]

print("="*70)
print("TESTING EXAMPLE REVIEWS")
print("="*70)

for i, review in enumerate(example_reviews, 1):
    print(f"\n{i}. Review: {review}")
    sentiment = predict_sentiment(review)
    print("-" * 70)

## Interactive Prediction

Enter your own reviews and see predictions in real-time!

In [ ]:
print("\n" + "="*70)
print("INTERACTIVE MODE")
print("="*70)
print("\nEnter a restaurant review (or 'quit' to exit):")
print("-" * 70)

while True:
    user_review = input("\nYour review: ").strip()
    
    if user_review.lower() in ['quit', 'exit', 'q', '']:
        print("\nExiting interactive mode.")
        break
    
    predict_sentiment(user_review)

## Aspect Analysis

Analyze which aspects are mentioned in reviews.

In [ ]:
def analyze_aspects(text):
    """
    Identify which aspects are mentioned in a review.
    
    Args:
        text (str): Review text
    
    Returns:
        list: Mentioned aspects
    """
    aspects = {
        'Food': ['food', 'dish', 'meal', 'taste', 'flavor', 'delicious', 'tasty', 'cuisine'],
        'Service': ['service', 'waiter', 'staff', 'server', 'friendly', 'attentive', 'rude'],
        'Ambiance': ['ambiance', 'atmosphere', 'decor', 'environment', 'cozy', 'noisy'],
        'Price': ['price', 'expensive', 'cheap', 'value', 'cost', 'affordable', 'overpriced']
    }
    
    text_lower = text.lower()
    mentioned_aspects = []
    
    for aspect, keywords in aspects.items():
        if any(keyword in text_lower for keyword in keywords):
            mentioned_aspects.append(aspect)
    
    return mentioned_aspects

# Test aspect analysis
print("\n" + "="*70)
print("ASPECT ANALYSIS")
print("="*70)

test_reviews = [
    "The food was amazing but the service was terrible.",
    "Great ambiance and reasonable prices.",
    "Expensive restaurant with slow service and mediocre food."
]

for review in test_reviews:
    print(f"\nReview: {review}")
    sentiment = predict_sentiment(review, show_confidence=False)
    aspects = analyze_aspects(review)
    print(f"  → Aspects mentioned: {', '.join(aspects) if aspects else 'None detected'}")
    print("-" * 70)

## Batch Prediction from CSV

In [ ]:
def batch_predict(csv_file, text_column='review'):
    """
    Predict sentiments for reviews in a CSV file.
    
    Args:
        csv_file (str): Path to CSV file
        text_column (str): Name of text column
    
    Returns:
        pd.DataFrame: DataFrame with predictions
    """
    print(f"Loading data from {csv_file}...")
    df = pd.read_csv(csv_file)
    print(f"✓ Loaded {len(df)} reviews")
    
    if text_column not in df.columns:
        print(f"❌ Error: Column '{text_column}' not found")
        print(f"Available columns: {list(df.columns)}")
        return None
    
    print("\nMaking predictions...")
    predictions = []
    
    for text in df[text_column]:
        pred = predict_sentiment(str(text), show_confidence=False)
        predictions.append(pred)
    
    df['predicted_sentiment'] = predictions
    
    print("\n✓ Predictions complete!")
    print("\nPrediction Summary:")
    for sentiment, count in pd.Series(predictions).value_counts().items():
        pct = count / len(predictions) * 100
        print(f"  {sentiment}: {count} ({pct:.1f}%)")
    
    return df

# Uncomment to use:
# result_df = batch_predict('your_reviews.csv', text_column='Review')
# result_df.to_csv('predictions_output.csv', index=False)
# print("\n✓ Results saved to 'predictions_output.csv'")

## Cross-Lingual Analysis (for XLM-RoBERTa)

Compare sentiment predictions across different languages.

In [ ]:
# Multilingual test reviews (same sentiment, different languages)
multilingual_reviews = {
    'English': "The food was excellent and the service was outstanding!",
    'French': "La nourriture était excellente et le service était exceptionnel!",
    'German': "Das Essen war ausgezeichnet und der Service war hervorragend!",
    'Spanish': "¡La comida fue excelente y el servicio fue sobresaliente!",
}

print("\n" + "="*70)
print("MULTILINGUAL SENTIMENT ANALYSIS")
print("="*70)
print("\nTesting the same positive sentiment across languages:\n")

for language, review in multilingual_reviews.items():
    print(f"{language}:")
    print(f"  Review: {review}")
    sentiment = predict_sentiment(review, show_confidence=True)
    print("-" * 70)

## Error Analysis

Examine cases where the model might struggle.

In [ ]:
# Challenging reviews (mixed sentiment, sarcasm, ambiguity)
challenging_reviews = [
    "The food was amazing but we waited 2 hours for our table.",
    "Great service, terrible food. What a disappointment.",
    "Oh sure, the best restaurant ever... if you like cold food and rude staff.",  # Sarcasm
    "It was okay. Nothing special.",  # Neutral/ambiguous
    "I really wanted to like this place, but...",  # Incomplete thought
]

print("\n" + "="*70)
print("CHALLENGING CASES - Error Analysis")
print("="*70)
print("\nTesting on ambiguous, mixed, or sarcastic reviews:\n")

for i, review in enumerate(challenging_reviews, 1):
    print(f"{i}. Review: {review}")
    sentiment = predict_sentiment(review, show_confidence=True)
    aspects = analyze_aspects(review)
    print(f"  → Aspects: {', '.join(aspects) if aspects else 'None'}")
    print("-" * 70)

## Visualize Predictions

In [ ]:
# Create sample predictions for visualization
all_test_reviews = example_reviews + test_reviews + challenging_reviews
all_predictions = [predict_sentiment(review, show_confidence=False) for review in all_test_reviews]

# Plot distribution
plt.figure(figsize=(10, 6))
sentiment_counts = pd.Series(all_predictions).value_counts()
colors = ['red' if s == 'NEGATIVE' else 'gray' if s == 'NEUTRAL' else 'green' for s in sentiment_counts.index]

sentiment_counts.plot(kind='bar', color=colors, alpha=0.7)
plt.title('Sentiment Distribution - Test Reviews', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal reviews analyzed: {len(all_test_reviews)}")

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print(f"MODEL: {MODEL_CHOICE.upper()}")
print("="*70)
print(f"\nPredictions Summary:")
print("-" * 70)
for sentiment, count in pd.Series(all_predictions).value_counts().items():
    pct = count / len(all_predictions) * 100
    print(f"  {sentiment:10}: {count:3} reviews ({pct:5.1f}%)")

print("\n" + "="*70)
print("✓ Analysis complete!")
print("="*70)

---

## 🎉 Prediction & Analysis Complete!

### What You've Done:
✅ Loaded trained model  
✅ Made predictions on various reviews  
✅ Analyzed aspect mentions  
✅ Tested multilingual capabilities (if XLM-RoBERTa)  
✅ Examined challenging cases  

### Key Insights for Your Report:
1. **Model Performance**: Document accuracy from Part 2
2. **Multilingual Capability**: XLM-RoBERTa handles multiple languages
3. **Aspect Detection**: Identifies key restaurant aspects
4. **Challenges**: Mixed sentiments and sarcasm remain difficult

### For Your Report:
- Use the visualizations from Part 2 (comparison charts, confusion matrices)
- Include examples from this notebook showing multilingual predictions
- Discuss the research gap you addressed (cross-cultural analysis)
- Highlight XLM-RoBERTa's superior performance on multilingual data

---